In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV

import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

feats = pd.read_csv("./user_features_pu.csv")
feature_cols = [c for c in feats.columns if c not in ('user_id','label_3way','is_sockpuppet','ip_checked')]
mask = feats['label_3way'].isin(['P','N'])
X = feats.loc[mask, feature_cols].to_numpy(np.float32)
y = (feats.loc[mask, 'label_3way'] == 'P').astype(int).to_numpy()

clfs = {
    'LogReg':    lambda: LogisticRegression(max_iter=2000, class_weight='balanced'),
    'RF':        lambda: RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1),
    'GBM':       lambda: GradientBoostingClassifier(n_estimators=200),
    'LinearSVM': lambda: CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=2000), cv=3),
}

for name, factory in clfs.items():
    f1s, aucs = [], []
    for seed in range(5):
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr, te in skf.split(X, y):
            sc = StandardScaler().fit(X[tr])
            clf = factory()
            clf.fit(sc.transform(X[tr]), y[tr])
            s = clf.predict_proba(sc.transform(X[te]))[:,1]
            p = (s >= 0.5).astype(int)
            f1s.append(f1_score(y[te], p, zero_division=0))
            aucs.append(roc_auc_score(y[te], s))
    print(f"{name:10s}  F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}  AUC={np.mean(aucs):.3f}±{np.std(aucs):.3f}")

LogReg      F1=0.552±0.047  AUC=0.685±0.051
RF          F1=0.440±0.078  AUC=0.646±0.052
GBM         F1=0.447±0.082  AUC=0.620±0.058


/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_class

LinearSVM   F1=0.105±0.097  AUC=0.676±0.054


/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_base.

In [2]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

feats = pd.read_csv("./user_features_pu.csv")
feature_cols = [c for c in feats.columns
                if c not in ('user_id','label_3way','is_sockpuppet','ip_checked')]
X = feats[feature_cols].to_numpy(np.float32)
labels = feats['label_3way'].values
P_mask, N_mask, U_mask = labels=='P', labels=='N', labels=='U'

# === D1 with LR ===
pn_mask = P_mask | N_mask
X_pn = X[pn_mask]
y_pn = P_mask[pn_mask].astype(int)

# CV metrics
f1s, recs, precs, aucs = [], [], [], []
for seed in range(5):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for tr, te in skf.split(X_pn, y_pn):
        sc = StandardScaler().fit(X_pn[tr])
        clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=seed)
        clf.fit(sc.transform(X_pn[tr]), y_pn[tr])
        s = clf.predict_proba(sc.transform(X_pn[te]))[:,1]
        p = (s >= 0.5).astype(int)
        f1s.append(f1_score(y_pn[te], p, zero_division=0))
        recs.append(recall_score(y_pn[te], p, zero_division=0))
        precs.append(precision_score(y_pn[te], p, zero_division=0))
        aucs.append(roc_auc_score(y_pn[te], s))
print(f"P-vs-N (LR):  F1={np.mean(f1s):.3f}  R={np.mean(recs):.3f}  P={np.mean(precs):.3f}  AUC={np.mean(aucs):.3f}")

# Apply to U
sc = StandardScaler().fit(X_pn)
clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=0)
clf.fit(sc.transform(X_pn), y_pn)
u_scores = clf.predict_proba(sc.transform(X[U_mask]))[:,1]

print("\nU-flagging at multiple thresholds:")
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    rate = (u_scores >= t).mean()
    print(f"  threshold {t:.1f}: {rate*100:5.1f}% of U flagged ({int(rate*U_mask.sum())} users)")

# === D4 with LR scores ===
u_indices = np.where(U_mask)[0]
top30_local = np.argsort(-u_scores)[:30]
top30_global = u_indices[top30_local]

profile = pd.DataFrame({
    'P_mean': feats.iloc[np.where(P_mask)[0]][feature_cols].mean(),
    'N_mean': feats.iloc[np.where(N_mask)[0]][feature_cols].mean(),
    'top_U_mean': feats.iloc[top30_global][feature_cols].mean(),
}).round(2)
profile['top_U_matches_P'] = (
    np.sign(profile['top_U_mean'] - profile['N_mean']) ==
    np.sign(profile['P_mean'] - profile['N_mean'])
)
matches = profile['top_U_matches_P'].sum()
print(f"\nD4 with LR: top-30 U matches P direction on {matches}/{len(profile)} features")
print(f"(Chance = {len(profile)//2})")
print(profile.to_string())

P-vs-N (LR):  F1=0.552  R=0.644  P=0.485  AUC=0.685

U-flagging at multiple thresholds:
  threshold 0.3:  65.0% of U flagged (441 users)
  threshold 0.4:  52.4% of U flagged (355 users)
  threshold 0.5:  35.4% of U flagged (240 users)
  threshold 0.6:  20.1% of U flagged (136 users)
  threshold 0.7:   9.1% of U flagged (62 users)
  threshold 0.8:   4.6% of U flagged (31 users)

D4 with LR: top-30 U matches P direction on 32/34 features
(Chance = 17)
                              P_mean     N_mean  top_U_mean  top_U_matches_P
n_posts                        16.83      12.17       41.10             True
n_threads                      11.98       9.37       21.70             True
days_active                     8.79       6.83       12.67             True
posts_per_thread                1.34       1.27        2.01             True
posts_per_day                  13.03      16.53        9.58             True
hour_entropy_bits               1.56       0.99        2.53             True
dayofwe